In [11]:
import pandas as pd
import json
import numpy as np


df_loai_da = pd.read_excel('data/LOAI_DA.xlsx')
df_nang_suat = pd.read_excel('data/NANG_SUAT.xlsx', header=2)
df_cong_nang = pd.read_excel('data/CONG_NANG_MAY.xlsx', header=2)
df_list_may = pd.read_excel('data/LIST_MAY.xlsx', header=1)

In [12]:
materials_map = {}

for index, row in df_loai_da.iterrows():
    mat_id = str(row['Material Number/ Mã vật tư'])
    group = str(row['NHÓM NĂNG SUẤT']).strip()

    # Chỉ lấy dữ liệu nếu không bị rỗng (NaN)
    if mat_id != 'nan' and group != 'nan':
        materials_map[mat_id] = group

# Kiểm tra thử 5 kết quả đầu
print(list(materials_map.items())[:10])

[('100000451598', 'C'), ('100000474067', 'A'), ('100000446895', 'A'), ('100000443841', 'C'), ('100000451698', 'F'), ('100000448092', 'A'), ('100000466614', 'C'), ('100000473282', 'C'), ('100001000455', 'C'), ('100000472537', 'A')]


In [13]:
df_nang_suat.rename(columns={
    df_nang_suat.columns[0]: 'STT',
    df_nang_suat.columns[1]: 'NHOM',
    df_nang_suat.columns[2]: 'KICH_THUOC'
}, inplace=True)

machine_columns = df_nang_suat.columns[3:]

df_melted = df_nang_suat.melt(
    id_vars=['NHOM', 'KICH_THUOC'],
    value_vars=machine_columns,
    var_name='MA_MAY',
    value_name='TOC_DO'
)

def clean_speed(val):
    try:
        clean_str = str(val).replace(',', '').replace(' ', '')
        return float(clean_str)
    except:
        return 0.0

df_melted['TOC_DO'] = df_melted['TOC_DO'].apply(clean_speed)

speed_matrix = {}
for _, row in df_melted.iterrows():
    m = row['MA_MAY']
    g = row['NHOM']
    s = row['KICH_THUOC']
    v = row['TOC_DO']

    if m not in speed_matrix: speed_matrix[m] = {}
    if g not in speed_matrix[m]: speed_matrix[m][g] = {}

    speed_matrix[m][g][s] = v

print("Đã xử lý xong ma trận năng suất!")

Đã xử lý xong ma trận năng suất!


In [14]:
# 1. Đổi tên cột Mã máy
df_cong_nang.rename(columns={df_cong_nang.columns[2]: 'MA_MAY'}, inplace=True)

# 2. Lấy danh sách các công việc
operation_cols = [c for c in df_cong_nang.columns[3:] if "Unnamed" not in str(c)]

machine_capabilities = {}

for _, row in df_cong_nang.iterrows():
    may_id = row['MA_MAY']
    if pd.isna(may_id): continue

    abilities = []
    for op in operation_cols:
        val = str(row[op]).upper()
        if 'X' in val:
            abilities.append(op)

    machine_capabilities[may_id] = abilities

print(machine_capabilities)

{'CNCCCC0001': ['Cắt thẳng', 'Chém 45'], 'CNCCCC0002': ['Cắt thẳng', 'Chém 45', 'Phào chỉ (CMS)'], 'CNCCCC0003': ['Cắt thẳng', 'Chém 45', 'Phào chỉ (CMS)'], 'CNCTNC0001': ['Cắt thẳng', 'Cắt dị hình'], 'CNCTNC0002': ['Cắt thẳng', 'Cắt dị hình'], 'CNCTNC0003': ['Cắt thẳng', 'Cắt dị hình'], 'CNCTNC0004': ['Cắt thẳng', 'Cắt dị hình'], 'CNCTNC0005': ['Cắt thẳng', 'Cắt dị hình'], 'CNCTRT0001': ['Chém 45', 'Phào chỉ (CMS)', 'VÁT LÁ HẸ', 'Đánh bóng '], 'CNCTRT0002': ['Chém 45', 'Phào chỉ (CMS)', 'VÁT LÁ HẸ', 'Đánh bóng '], 'CATVTC0001': ['Chém 45'], 'CNCTNC0006': ['Cắt thẳng', 'Cắt dị hình'], 'CNCKIH0001': ['Cắt thẳng'], 'CATVTC0002': ['Đánh bóng '], 'CATVTC0003': ['VÁT LÁ HẸ'], 'MAIKTR0001': ['Đánh bóng '], 'VSHRKH0001': [], 'KHNKNH0001': []}


In [15]:
final_data = {
    "materials_map": materials_map,
    "machines": {}
}

all_machine_ids = set(speed_matrix.keys()) | set(machine_capabilities.keys())

for m_id in all_machine_ids:
    final_data["machines"][m_id] = {
        "capabilities": machine_capabilities.get(m_id, []),
        "speed_matrix": speed_matrix.get(m_id, {})
    }

with open('cleaned_master_data.json', 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("Hoàn tất! File cleaned_master_data.json đã sẵn sàng.")

Hoàn tất! File cleaned_master_data.json đã sẵn sàng.
